In [ ]:
import pandas as pd
import os
import snowflake.connector
from snowflake.connector.pandas_tools import write_pandas
import spacy
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

# connect to database
SF_USR =os.getenv('SF_USR')
SF_KEY =os.getenv('SF_KEY')
SF_ID  =os.getenv('SF_ID')
SF_WH  =os.getenv('SF_WH')
SF_DB  =os.getenv('SF_DB')
SF_SC  =os.getenv('SF_SC')
SF_RL  =os.getenv('SF_RL')
SF_XCT = snowflake.connector.connect(
    user      = SF_USR
   ,password  = SF_KEY
   ,account   = SF_ID
   ,warehouse = SF_WH
   ,database  = SF_DB
   ,schema    = SF_SC
   ,role      = SF_RL
)
CSR = SF_XCT.cursor()

# Initialize VADER
SENTANA = SentimentIntensityAnalyzer()
def get_vader_sentiment(post_text: str) -> dict:
    """
    Return the VADER sentiment scores for the text in a social media post.
    Read more about VADER here:
        https://github.com/cjhutto/vaderSentiment
    
    Args:
        analyzer: A VADER SentimentIntesnityAnalyzer object; this contains the methods
                  needed to compute the sentiment of a post.
        post_text: The raw text from a social media post. 
    """
    return SENTANA.polarity_scores(post_text)

# get data from snowflake
query = f"""
select content_id
      ,usa_timestamp
      ,detected_languages
      ,post_text 
from {SF_DB}.main.firehose_processed
where detected_languages ilike '%en%';
"""
CSR.execute(query)
data = CSR.fetch_pandas_all()
data['VADER_SENTIMENT_SCORE'] = data['POST_TEXT'].apply(get_vader_sentiment)
write_pandas(SF_XCT, df=data, table_name='POSTS_VADER_SENTIMENT', database=SF_DB, schema=SF_SC)


,CONTENT_ID,USA_TIMESTAMP,DETECTED_LANGUAGES,POST_TEXT,VADER_SENTIMENT_SCORE
0,bafyreiavhrixpeyeoktype2bdn5fbykvz4pcpvi673l2w...,2025-05-24 13:19:18.331774+00:00,"[""en""]",Is there any way to transfer tweets from twitt...,"{'neg': 0.0, 'neu': 1.0, 'pos': 0.0, 'compound..."
1,bafyreifp2exx72li3fdu56hm3tzwnfhoi7ecmeproxr4h...,2025-05-24 03:02:16.286972+00:00,"[""en""]",Final:\nMariners 5\nAstros 3,"{'neg': 0.0, 'neu': 1.0, 'pos': 0.0, 'compound..."
2,bafyreidyb5vqa52swxozkhghdbcdzhudoimsmbvpltpxe...,2025-05-24 16:20:01.300524+00:00,"[""en""]",www.bbc.com/news/article...,"{'neg': 0.0, 'neu': 1.0, 'pos': 0.0, 'compound..."
3,bafyreifvj2jld664f37er6mso3zhvbrqz2vq6enekk5kh...,2025-05-24 16:19:18.723037+00:00,"[""en""]",A stud will look you dead in your face and say...,"{'neg': 0.236, 'neu': 0.643, 'pos': 0.121, 'co..."
4,bafyreicxmv7o6emv53vz7lrjwxfitoygeebq2trekmwuk...,2025-05-24 16:18:07.904736+00:00,"[""en""]",I've been watching your videos for almost ten ...,"{'neg': 0.039, 'neu': 0.642, 'pos': 0.319, 'co..."


In [21]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

analyzer = SentimentIntensityAnalyzer()

s = "QUALITY!!!!"
analyzer.polarity_scores(s)

{'neg': 0.0, 'neu': 1.0, 'pos': 0.0, 'compound': 0.0}